In [1]:
import numpy as np
import pandas as pd
import scipy.linalg as la
import cvxpy as cvx

## Homework 5

Problem 1: Verify that the matrix Q is symmetric and positive semidefinite. Then solve the optimization problem.

In [2]:
Q = np.array([[4.,1,1],[1,2,0],[1,0,1]])
c = np.array([1.,-2,3])
a = np.ones(3)
print(Q)
print(c)
print(a)

[[4. 1. 1.]
 [1. 2. 0.]
 [1. 0. 1.]]
[ 1. -2.  3.]
[1. 1. 1.]


In [3]:
print("symmetric:", np.allclose(Q,Q.T))

symmetric: True


In [4]:
print("  eigs:", np.linalg.eigvalsh(Q), " PSD:", np.all(np.linalg.eigvalsh(Q) >= 0))

  eigs: [0.62279715 1.72610945 4.65109341]  PSD: True


In [5]:
M=np.block([[2*Q, a.reshape(3,1)],[a.reshape(1,3), np.zeros((1,1))]])
rhs= np.concatenate([-c,[1]])
sol=la.solve(M,rhs, assume_a='sym')
x, z = sol[:3], sol[3]

In [6]:
print(x, z, x@Q@x + c@x) 
print("feasible :", a@x - 1)
print("stationary:", 2*Q@x + c + z*a)

[-0.0625  1.1875 -0.125 ] -2.625 -0.09375
feasible : 0.0
stationary: [0. 0. 0.]


Problem 2) Let Q and c be as in the previous problem.Solve the optimization problem Hint: Before applying the  methods recently covered inclass, think about what the
feasible region looks like.This problem is easier than the previous one.

In [7]:
A2 = np.array([[1.,1,1],[0,1,1],[0,0,1]])
b2 = np.zeros(3)
x2 = la.solve(A2, b2)
print("det A:", np.linalg.det(A2), " x*:", x2, " f(x*):", x2@Q@x2 + c@x2)

det A: 1.0  x*: [0. 0. 0.]  f(x*): 0.0


Problem 3: Consider a variant of the Markowitz portfolio optimization problemin which, instead
of requiring that the expected return i sbounded below by a constant α, we require that the expected return is exactly equal to α.Revisit the stockmarket data from the introduction to Markowitz portfolio optimization(the Portfolio Optimization Complete) notebookonSakai)and solve this problem as an equality-constrained quadratic program.
Inparticular, if Σ denotes the estimated covariance matrix of asset returns,r denotes the vector of estimated expected returns,and x denotes the vector of portfolio weights,
Report the optimal portfolio weights and the corresponding portfolio risk, and verify
that both equality constraints are satisfied.

In [8]:
df= pd.read_csv("DailyReturns.csv", index_col=0)

In [9]:
Sigma=df.cov().to_numpy()
r=df.mean().to_numpy()
n=df.shape[1]
alpha = 0.05 
print(r"sigma=", Sigma)
print(r"mean=", r)
print(r"num_assets=", n)

sigma= [[ 2.47734349  1.727213    1.33683046  1.19623842  1.48428619  3.10979589
   3.01886918  0.45913749  0.89471849]
 [ 1.727213    3.20826405  1.15006071  1.5115829   1.87614091  4.28563425
   3.4436514   0.58177673  1.05767218]
 [ 1.33683046  1.15006071  5.21966944  1.04034305  1.99521369  2.34979949
   2.41930885  0.14913676  0.58856258]
 [ 1.19623842  1.5115829   1.04034305  1.55522523  1.17608826  2.96481601
   2.58775758  0.36487915  0.68181899]
 [ 1.48428619  1.87614091  1.99521369  1.17608826 19.07791673  4.60378344
   3.58275768  1.00767967  1.17468651]
 [ 3.10979589  4.28563425  2.34979949  2.96481601  4.60378344 15.17913833
   6.73771813  1.52829891  2.33904462]
 [ 3.01886918  3.4436514   2.41930885  2.58775758  3.58275768  6.73771813
   7.99446348  1.01136861  1.69269412]
 [ 0.45913749  0.58177673  0.14913676  0.36487915  1.00767967  1.52829891
   1.01136861  0.70966071  0.63555075]
 [ 0.89471849  1.05767218  0.58856258  0.68181899  1.17468651  2.33904462
   1.69269412  

In [10]:
A=np.vstack([r, np.ones(n)])
b=np.array([alpha,1])
M=np.block([[2*Sigma, A.T], [A, np.zeros((2,2))]])
rhs=np.concatenate([np.zeros(n), b])
sol=la.solve(M, rhs, assume_a='sym')
xk=sol[:n]

In [11]:
xk

array([ 0.26261686, -0.15459203, -0.00439645,  0.56462671,  0.00848279,
       -0.04591242, -0.17226198,  0.66549332, -0.12405679])

In [12]:
print(pd.Series(xk, index=df.columns).round(4))       
print("portfolio risk :", xk @ Sigma @ xk)                 
print("std dev    :", np.sqrt(xk @ Sigma @ xk))
print("return chk :", r @ xk, "vs", alpha)               
print("budget chk :", xk.sum(), "vs 1")                 


AAPL     0.2626
AMZN    -0.1546
BRK-B   -0.0044
GOOG     0.5646
LLY      0.0085
META    -0.0459
MSFT    -0.1723
NVDA     0.6655
TSM     -0.1241
dtype: float64
portfolio risk : 0.5253133357907974
std dev    : 0.7247850272948506
return chk : 0.05 vs 0.05
budget chk : 1.0 vs 1


In [13]:
print("rank A      :", np.linalg.matrix_rank(A), "of", A.shape[0])
resid = A @ xk - b
print("A x - b     :", resid)
print("max |resid| :", np.abs(resid).max())

rank A      : 2 of 2
A x - b     : [ 0.00000000e+00 -1.11022302e-16]
max |resid| : 1.1102230246251565e-16
